# So sánh dataset đã tạo với paper DeepSMOTE

Notebook này **chỉ đọc** dataset đã có trong `data/processed/` (tạo bởi `src/preprocessing/create_imbalanced_dataset.py`) và dùng lại các hàm trong `src/visualization/report.py` để so sánh số mẫu mỗi class với số liệu paper DeepSMOTE (`DATASET_CONFIGS`, lấy từ mảng `imbal` trong paper / `DeepSMOTE/GenerateSamples.py`).

Notebook không sample lại, không train, không sửa dữ liệu — chỉ hiển thị kết quả.

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from visualization.report import (
    load_processed_dataset,
    compare_with_deepsmote,
    comparison_all_match,
    format_comparison_table,
    plot_class_distribution,
    plot_sample_grid,
)

## Chọn dataset và seed

Đổi `DATASET` thành một trong: `mnist`, `fashion_mnist`, `cifar10`, `svhn`, `celeba` (phải chạy `create_imbalanced_dataset.py` cho dataset đó trước).

In [ ]:
DATASET = "mnist"
SEED = 42

ds = load_processed_dataset(DATASET, SEED)
print(f"Đã load {ds.cfg.display_name}: train={len(ds.train_y)} mẫu, test={len(ds.test_y)} mẫu")

## Bảng so sánh số mẫu mỗi class: paper DeepSMOTE vs. dataset đã tạo

In [ ]:
rows = compare_with_deepsmote(ds)
print(format_comparison_table(rows))
print()
print("Giống paper DeepSMOTE:", "CÓ" if comparison_all_match(rows) else "KHÔNG")

## Biểu đồ phân bố class (paper vs. thực tế)

In [ ]:
fig = plot_class_distribution(ds, rows)

## Lưới ảnh mẫu theo từng class

In [ ]:
fig = plot_sample_grid(ds, samples_per_class=6, seed=SEED)

## So sánh nhanh tất cả dataset đã tạo

Chạy cell dưới để kiểm tra cùng lúc mọi dataset đã có trong `data/processed/` (dataset chưa tạo sẽ báo lỗi "chưa thấy dataset" và bị bỏ qua).

In [ ]:
from preprocessing.config import DATASET_CONFIGS

for key in DATASET_CONFIGS:
    try:
        ds_i = load_processed_dataset(key, SEED)
    except FileNotFoundError as exc:
        print(f"{key:16s} -> {exc}")
        continue
    rows_i = compare_with_deepsmote(ds_i)
    status = "CÓ" if comparison_all_match(rows_i) else "KHÔNG"
    print(f"{key:16s} -> giống paper DeepSMOTE: {status}")